# Query 3: Total Revenue Grouped by Hour of Day + VendorID
**Type:** Grouping by Multiple Attributes  
**Problem Statement:** Calculate total revenue and trip count grouped by both hour of day and VendorID. This reveals which vendors are busiest and most profitable at each hour.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('Q3_GroupBy') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.eventLog.enabled', 'false') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 19:49:04 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 19:49:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 19:49:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


In [2]:
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
          .csv('../data/yellow_tripdata_2015-01.csv') \
          .sample(fraction=0.2, seed=42)

df = df.withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
       .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
       .withColumnRenamed('fare_amount',            'fare') \
       .withColumnRenamed('passenger_count',        'passengers') \
       .withColumnRenamed('trip_distance',          'distance') \
       .withColumnRenamed('total_amount',           'total')

df = df.withColumn('fare',       F.col('fare').cast(DoubleType())) \
       .withColumn('total',      F.col('total').cast(DoubleType())) \
       .withColumn('distance',   F.col('distance').cast(DoubleType())) \
       .withColumn('passengers', F.col('passengers').cast(IntegerType())) \
       .cache()

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows (20% sample):', df.count())

Total rows (20% sample): 2233826


## RDD Implementation

In [3]:
start = time.time()

result_rdd = (
    rdd
    .filter(lambda r: r['pickup_datetime'] is not None
                  and r['fare']            is not None)
    .map(lambda r: (
        (r['pickup_datetime'].hour, r['VendorID']),
        r['fare']
    ))
    .reduceByKey(lambda a, b: a + b)
    .sortBy(lambda x: -x[1])
    .take(20)
)

rdd_time = time.time() - start
print(f'RDD | Time: {rdd_time:.2f}s')
print('Top 20 hour+vendor by revenue (RDD):')
for (hr, vendor), rev in result_rdd:
    print(f'  hour={hr}  vendor={vendor}  revenue=${rev:,.2f}')

RDD | Time: 129.17s
Top 20 hour+vendor by revenue (RDD):
  hour=18  vendor=2  revenue=$851,601.36
  hour=19  vendor=2  revenue=$844,515.17
  hour=21  vendor=2  revenue=$784,356.00
  hour=22  vendor=2  revenue=$779,961.83
  hour=20  vendor=2  revenue=$767,249.82
  hour=17  vendor=2  revenue=$749,403.38
  hour=14  vendor=2  revenue=$742,237.48
  hour=18  vendor=1  revenue=$732,906.96
  hour=15  vendor=2  revenue=$732,310.66
  hour=19  vendor=1  revenue=$726,095.05
  hour=21  vendor=1  revenue=$693,389.02
  hour=20  vendor=1  revenue=$691,713.56
  hour=22  vendor=1  revenue=$690,037.67
  hour=13  vendor=2  revenue=$683,330.41
  hour=23  vendor=2  revenue=$680,912.60
  hour=12  vendor=2  revenue=$668,077.96
  hour=14  vendor=1  revenue=$655,690.27
  hour=16  vendor=2  revenue=$655,165.90
  hour=15  vendor=1  revenue=$653,883.95
  hour=11  vendor=2  revenue=$629,519.00


## DataFrame Implementation

In [4]:
start = time.time()

result_df = (
    df.withColumn('pickup_hour', F.hour('pickup_datetime'))
      .groupBy('pickup_hour', 'VendorID')
      .agg(
          F.round(F.sum('fare'), 2) .alias('total_revenue'),
          F.count('*')              .alias('trip_count'),
          F.round(F.avg('fare'), 2) .alias('avg_fare')
      )
      .orderBy(F.desc('total_revenue'))
)

print('--- Q3 DataFrame explain(True) ---')
result_df.explain(True)
result_df.cache()
df_time = time.time() - start
print(f'DataFrame | Time: {df_time:.2f}s')
result_df.show(20)
result_df.unpersist()

--- Q3 DataFrame explain(True) ---
== Parsed Logical Plan ==
'Sort ['total_revenue DESC NULLS LAST], true
+- Aggregate [pickup_hour#1249, VendorID#17], [pickup_hour#1249, VendorID#17, round(sum(fare#176), 2) AS total_revenue#1291, count(1) AS trip_count#1293L, round(avg(fare#176), 2) AS avg_fare#1295]
   +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total#196, hour(pickup_datetime#55, Some(America/New_York)) AS pickup_hour#1249]
      +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, cast(passengers#116 as int) AS passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#

+-----------+--------+-------------+----------+--------+
|pickup_hour|VendorID|total_revenue|trip_count|avg_fare|
+-----------+--------+-------------+----------+--------+
|         18|       2|    851601.36|     75273|   11.31|
|         19|       2|    844515.17|     75913|   11.12|
|         21|       2|     784356.0|     65681|   11.94|
|         22|       2|    779961.83|     63018|   12.38|
|         20|       2|    767249.82|     66718|    11.5|
|         17|       2|    749403.38|     63656|   11.77|
|         14|       2|    742237.48|     60325|    12.3|
|         18|       1|    732906.96|     65712|   11.15|
|         15|       2|    732310.66|     58862|   12.44|
|         19|       1|    726095.05|     65820|   11.03|
|         21|       1|    693389.02|     58905|   11.77|
|         20|       1|    691713.56|     60770|   11.38|
|         22|       1|    690037.67|     56944|   12.12|
|         13|       2|    683330.41|     58168|   11.75|
|         23|       2|     6809

DataFrame[pickup_hour: int, VendorID: int, total_revenue: double, trip_count: bigint, avg_fare: double]

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT   HOUR(pickup_datetime)        AS pickup_hour,
             VendorID,
             ROUND(SUM(fare),  2)         AS total_revenue,
             COUNT(*)                     AS trip_count,
             ROUND(AVG(fare),  2)         AS avg_fare
    FROM     trips
    GROUP BY HOUR(pickup_datetime), VendorID
    ORDER BY total_revenue DESC
    LIMIT    20
""")

print('--- Q3 Spark SQL explain(True) ---')
result_sql.explain(True)
result_sql.cache()
sql_time = time.time() - start
print(f'SQL | Time: {sql_time:.2f}s')
result_sql.show()
result_sql.unpersist()

--- Q3 Spark SQL explain(True) ---
== Parsed Logical Plan ==
'GlobalLimit 20
+- 'LocalLimit 20
   +- 'Sort ['total_revenue DESC NULLS LAST], true
      +- 'Aggregate ['HOUR('pickup_datetime), 'VendorID], ['HOUR('pickup_datetime) AS pickup_hour#2230, 'VendorID, 'ROUND('SUM('fare), 2) AS total_revenue#2231, 'COUNT(1) AS trip_count#2232, 'ROUND('AVG('fare), 2) AS avg_fare#2233]
         +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
pickup_hour: int, VendorID: int, total_revenue: double, trip_count: bigint, avg_fare: double
GlobalLimit 20
+- LocalLimit 20
   +- Sort [total_revenue#2231 DESC NULLS LAST], true
      +- Aggregate [hour(pickup_datetime#55, Some(America/New_York)), VendorID#17], [hour(pickup_datetime#55, Some(America/New_York)) AS pickup_hour#2230, VendorID#17, round(sum(fare#176), 2) AS total_revenue#2231, count(1) AS trip_count#2232L, round(avg(fare#176), 2) AS avg_fare#2233]
         +- SubqueryAlias trips
            +- View (`trips`, [VendorID#17,p

+-----------+--------+-------------+----------+--------+
|pickup_hour|VendorID|total_revenue|trip_count|avg_fare|
+-----------+--------+-------------+----------+--------+
|         18|       2|    851601.36|     75273|   11.31|
|         19|       2|    844515.17|     75913|   11.12|
|         21|       2|     784356.0|     65681|   11.94|
|         22|       2|    779961.83|     63018|   12.38|
|         20|       2|    767249.82|     66718|    11.5|
|         17|       2|    749403.38|     63656|   11.77|
|         14|       2|    742237.48|     60325|    12.3|
|         18|       1|    732906.96|     65712|   11.15|
|         15|       2|    732310.66|     58862|   12.44|
|         19|       1|    726095.05|     65820|   11.03|
|         21|       1|    693389.02|     58905|   11.77|
|         20|       1|    691713.56|     60770|   11.38|
|         22|       1|    690037.67|     56944|   12.12|
|         13|       2|    683330.41|     58168|   11.75|
|         23|       2|     6809

DataFrame[pickup_hour: int, VendorID: int, total_revenue: double, trip_count: bigint, avg_fare: double]

## Performance Comparison

In [6]:
print('='*65)
row1 = f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}'
row2 = f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s'
row3 = f'{"Grouping Keys":<25} {"(hour,vendor)":>12} {"(hour,vendor)":>12} {"(hour,vendor)":>12}'
row4 = f'{"Optimizer":<25} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}'
print(row1)
print('-'*65)
print(row2)
print(row3)
print(row4)
print('='*65)
print()
print('KEY INSIGHT:')
print('Multi-key GROUP BY requires composite key tuples in RDD.')
print('DataFrame/SQL handle multi-column grouping natively.')
print('Catalyst generates HashAggregate with optimized partial aggregation.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                 129.17s        0.33s        0.69s
Grouping Keys             (hour,vendor) (hour,vendor) (hour,vendor)
Optimizer                         None     Catalyst     Catalyst

KEY INSIGHT:
Multi-key GROUP BY requires composite key tuples in RDD.
DataFrame/SQL handle multi-column grouping natively.
Catalyst generates HashAggregate with optimized partial aggregation.
